In [ ]:
!pip install -U transformers datasets[audio] accelerate evaluate jiwer
!pip install sentencepiece librosa soundfile

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!unzip dataset.zip

In [ ]:
!find /content -name "0001.wav"

In [ ]:
import pandas as pd

df = pd.read_csv("/content/dataset/metadata.csv")

print(df.columns)
print(df.head())

In [ ]:
import pandas as pd

df = pd.read_csv("/content/dataset/metadata.csv")

df["audio_path"] = df["audio_path"].apply(
    lambda x: "/content/dataset/audio/" + x.split("/")[-1]
)

df.to_csv(
    "/content/dataset/metadata_fixed.csv",
    index=False
)

print(df.head())

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="/content/dataset/metadata_fixed.csv"
)

In [ ]:
from datasets import Audio

dataset = dataset.cast_column(
    "audio_path",
    Audio(sampling_rate=16000)
)

In [ ]:
MODEL_NAME = "openai/whisper-tiny"

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language="russian",
    task="transcribe"
)

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME
)

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio_path"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(
        batch["text"]
    ).input_ids

    return batch

In [ ]:
dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names
)

In [ ]:
!find /content -name "0001.wav"

In [ ]:
dataset = dataset["train"].train_test_split(
    test_size=0.2,
    seed=42
)

In [ ]:
print(dataset)

In [ ]:
!pip install evaluate jiwer

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")

In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = processor.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {"wer": wer}

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-tiny-maskarad",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=1e-5,

    warmup_steps=10,

    max_steps=100,

    eval_strategy="steps",
    eval_steps=10,

    save_steps=10,
    save_total_limit=3,

    logging_steps=10,

    predict_with_generate=True,

    fp16=True,

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

In [ ]:
import torch

def data_collator(features):
    # input_features (audio)
    input_features = [{"input_features": f["input_features"]} for f in features]
    batch = processor.feature_extractor.pad(
        input_features,
        return_tensors="pt"
    )

    # labels (text)
    label_features = [{"input_ids": f["labels"]} for f in features]
    labels_batch = processor.tokenizer.pad(
        label_features,
        return_tensors="pt"
    )

    labels = labels_batch["input_ids"]

    # важно: padding токены → -100 (игнор в loss)
    labels = labels.masked_fill(
        labels_batch["attention_mask"] != 1,
        -100
    )

    batch["labels"] = labels

    return batch

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
print(trainer.state.best_model_checkpoint)
print(trainer.state.best_metric)

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(
    "/content/whisper-tiny-maskarad/checkpoint-30"
)

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from transformers import WhisperForConditionalGeneration

model_path = "./whisper-tiny-maskarad/checkpoint-30"

model = WhisperForConditionalGeneration.from_pretrained(model_path)

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-tiny",
    language="russian",
    task="transcribe"
)

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
repo_name = "whisper-tiny-maskarad-30"

In [ ]:
model.push_to_hub(repo_name)
processor.push_to_hub(repo_name)